In [29]:
import duckdb
import pandas as pd
import os
from datetime import datetime

In [30]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [39]:
arquivo = 'z0019_1.csv'
data_ingestao = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
df = pd.read_csv(f'../landing/{arquivo}', sep=';')
df['nome_arquivo'] = arquivo
df['data_ingestao'] = data_ingestao
df.head()

,NARBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,150,z0019_1.csv,2025-12-04 16:48:29
1,10002,PORCA,BT20,100,250,z0019_1.csv,2025-12-04 16:48:29
2,10003,PREGO,BT30,100,50,z0019_1.csv,2025-12-04 16:48:29
3,10004,SERRA,BT40,100,200,z0019_1.csv,2025-12-04 16:48:29


In [32]:
#ingestão dos arquivos na tabela bronze 
con .execute("""
    CREATE TABLE IF NOT EXISTS bronze_z0019 (
        NARBR VARCHAR,
        MAKTX VARCHAR,
        WERKS VARCHAR,
        MAINS VARCHAR,
        LABST VARCHAR,
        nome_arquivo VARCHAR,
        data_ingestao TIMESTAMP)
""")

In [40]:
con.execute("Alter table bronze_z0019 add column if not exists nome_arquivo VARCHAR")
con.execute("Alter table bronze_z0019 add column if not exists data_ingestao TIMESTAMP")
con.execute("select * from bronze_z0019 limit 10").df()

,NARBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao
0,10001,PARAFUSO,BT10,100,150,None,NaT
1,10002,PORCA,BT20,100,250,None,NaT
2,10003,PREGO,BT30,100,50,None,NaT
3,10004,SERRA,BT40,100,200,None,NaT
4,10003,PREGO,BT90,100,50,z0019_2.csv,2025-12-04 16:45:24
5,10004,SERRA,BT50,100,200,z0019_2.csv,2025-12-04 16:45:24
6,10005,MACHADO,BT60,100,100,z0019_2.csv,2025-12-04 16:45:24
7,10003,PREGO,BT200,100,50,z0019_2.csv,2025-12-04 16:45:24
8,10009,CHAVE DE FENDA,BT70,100,300,z0019_2.csv,2025-12-04 16:45:24


In [38]:
#Inserção dos dados do dataframe na tabela bronze
con.execute("Insert into bronze_z0019 select * from df")

In [26]:
#validar conteúdo da tabela bronze
resultado = con.execute("select * from bronze_z0019").fetchdf()
print(resultado)

   NARBR     MAKTX WERKS  MAINS  LABST
0  10001  PARAFUSO  BT10    100    150
1  10002     PORCA  BT20    100    250
2  10003     PREGO  BT30    100     50
3  10004     SERRA  BT40    100    200


In [ ]:
#con.execute("alter table bronze_produtos rename to bronze_z0019")

In [ ]:
con.close()

In [27]:
# ⚠️ Certifique-se que o caminho do CSV ainda está correto!
caminho_csv_1 = '../landing/z0019_1.csv' 

con.execute(f"""
    -- CREATE OR REPLACE TABLE GARANTE QUE A TABELA É CRIADA NO DB
    CREATE OR REPLACE TABLE bronze_z0019 AS 
    SELECT * FROM read_csv_auto('{caminho_csv_1}', sep=';')
""")

# Recomendado: Feche a conexão para forçar o DuckDB a gravar o arquivo no disco
con.close()
print("Tabela bronze_z0019 criada e a conexão foi fechada (Persistida).")

Tabela bronze_z0019 criada e a conexão foi fechada (Persistida).
